# Dealing with Mysterious Conflicts

This notebook shows *reduce/reduce* conflicts that result from the fact that the set of `LR` states is
compressed into the set of `LALR` states.  Such a conflict is called *mysterious*, because it is not caused
by any defect of the grammar:  the grammar below is unambiguous and it even has the `LR(1)` property.  The
conflict appears only when two `LR` states that have the same *core* are merged into a single `LALR` state.

We discuss the following grammar:
```
    s : "v" a "y"
      | "w" b "y"
      | "v" b "z"
      | "w" a "z"

    a : X

    b : X

    X : "x"
```
The language generated by this grammar consists of the four strings `vxy`, `vxz`, `wxy` and `wxz`.  Whether
the `x` in the middle is an `a` or a `b` is determined by the *combination* of the first and the last
character:  in `vxy` and `wxz` it is an `a`, while in `wxy` and `vxz` it is a `b`.

## Specification of the Grammar

The tokens are defined by the scanner specification for `ocamllex` and the grammar rules are given in the grammar for
`Menhir`.  (The Python version of this notebook uses `Lark`, where both are given in a single grammar.)  In the `Lark`
grammar, the characters `v`, `w`, `y` and `z` occur as *anonymous terminals*, that is, as string literals inside the
rules, and `Lark` invents the names `V`, `W`, `Y` and `Z` for them.  We use the same names for the tokens, and these
names show up in the error message below.  Every token carries a value of type `Parse_tree.tok`, so that it can be
kept in the parse tree.

In [ ]:
let lexer = {|
{
  open S_parser
}

rule token = parse
  | [' ' '\t' '\r' '\n']+        { token lexbuf }      (* %ignore WS *)
  | 'v'                          { V (Parse_tree.token "V" lexbuf) }
  | 'w'                          { W (Parse_tree.token "W" lexbuf) }
  | 'y'                          { Y (Parse_tree.token "Y" lexbuf) }
  | 'z'                          { Z (Parse_tree.token "Z" lexbuf) }
  | 'x'                          { X (Parse_tree.token "X" lexbuf) }
  | eof                          { EOF }
  | _                            { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ repr (Lexing.lexeme lexbuf))) }
|}

let grammar = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> V W X Y Z
%token EOF

%start <Parse_tree.tree> start
%type  <Parse_tree.tree> s a b

%%

start: t = s EOF               { t }

s : v = V x = a y = Y          { Tree ("s", [Token v; x; Token y]) }
  | w = W x = b y = Y          { Tree ("s", [Token w; x; Token y]) }
  | v = V x = b z = Z          { Tree ("s", [Token v; x; Token z]) }
  | w = W x = a z = Z          { Tree ("s", [Token w; x; Token z]) }

a : x = X                      { Tree ("a", [Token x]) }

b : x = X                      { Tree ("b", [Token x]) }
|}

## Trying to Build an `LALR` Parser

`Menhir` prints its warnings about conflicts, and with the option `--explain` it writes a detailed explanation of every
conflict into the file `_build/s_parser.conflicts`.  The option `--strict` turns the warnings into errors, so that
`Menhir` refuses to build a parser with conflicts, like `Lark` does for *reduce/reduce* conflicts.

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
open Parse_tree

In [ ]:
try
  generate_parser_with ~options:"--lalr --strict" ~name:"s" ~lexer ~grammar;
  print_endline "No conflict."
with Grammar_error e ->
  print_string e;
  cat "_build/s_parser.conflicts"

## Where Does the Conflict Come From?

If we compute the `LR(1)` states of this grammar, we find the two states
$$ \{\; a \rightarrow X \bullet : \texttt{'y'}, \quad b \rightarrow X \bullet : \texttt{'z'} \;\}
   \qquad\text{and}\qquad
   \{\; b \rightarrow X \bullet : \texttt{'y'}, \quad a \rightarrow X \bullet : \texttt{'z'} \;\} $$
The first of these states is reached after reading `vx`, the second one after reading `wx`.

Taken by itself, *neither* of these two states has a conflict, since the follow sets of the two rules are
disjoint in both cases.  This is precisely the reason why the grammar has the `LR(1)` property.

However, the two states have the same *core*
$$ \{\; a \rightarrow X \bullet, \quad b \rightarrow X \bullet \;\} $$
and therefore they are merged when the `LALR` table is computed.  The resulting state is
$$ \{\; a \rightarrow X \bullet : \{\texttt{'y'}, \texttt{'z'}\}, \quad
       b \rightarrow X \bullet : \{\texttt{'y'}, \texttt{'z'}\} \;\} $$
and this state obviously has a *reduce/reduce* conflict, both for the token `'y'` and for the token `'z'`.
This is the conflict reported above.  `Menhir` tries to explain every conflict by two derivations, but here it
fails:  it calls the conflict *unexplainable* and remarks that it may be an *artificial* conflict caused by the option
`--lalr`.  The file `_build/s_parser.automaton` shows the merged state together with its follow sets.

The merger has thrown away the information about which of the two states we came from, and it is precisely
this information that would have been needed to decide between `a` and `b`.  The grammar is therefore an
`LR(1)` grammar, but *not* an `LALR(1)` grammar.

Since `Menhir` has refused to build the parser, there is no parser that we could use.

## The Real Solution: An `LR(1)` Parser

`Lark` can only build `LALR(1)` parsers.  `Menhir` can do better:  with the option `--canonical`, it builds the
*canonical* `LR(1)` automaton of Knuth, whose states are never merged.  As the grammar has the `LR(1)` property, this
automaton has no conflicts.  (Without one of the options `--lalr` and `--canonical`, `Menhir` uses the algorithm of
*Pager*, which merges two states only if the merger does not create a conflict.  Hence, it would work, too.)
The price is that the canonical `LR(1)` automaton usually has many more states than the `LALR(1)` automaton.

Since no warning is printed below, the grammar is an `LR(1)` grammar.  The file `_build/s_parser.automaton.resolved`
contains the two states discussed above:  one of them is reached after reading `v x`, the other one after
reading `w x`.

In [ ]:
generate_parser_with ~options:"--canonical --strict" ~name:"s" ~lexer ~grammar

In [ ]:
List.iter (fun s -> print_endline (pretty (parse_string S_parser.start S_lexer.token s))) [ "vxy"; "vxz"; "wxy"; "wxz" ]

## Another Way Out: A `GLR` Parser

The *default* parser of `Lark` is an `Earley` parser and the `Earley` algorithm does not build a parse table
at all.  The counterpart of `Menhir` is a *generalized LR parser*, which is created with the option `--GLR`.  In a state
with a conflict, a `GLR` parser pursues all possible actions in parallel.  Therefore, the grammar is accepted as it
stands and all four strings of the language are parsed correctly.  `Menhir` warns that a *merge function* should be
provided.  A merge function decides which parse tree is returned if a string has more than one parse tree.  Since our
grammar is unambiguous, we need no merge function.

The `Lark` version needs the keyword argument `keep_all_tokens=True` in order to keep the anonymous terminals in the
parse tree.  In `Menhir`, the semantic actions put the tokens into the tree.

The price for this convenience is a parser that is considerably slower than an `LALR` parser.

In [ ]:
generate_parser_with ~options:"--GLR" ~name:"s" ~lexer ~grammar

In [ ]:
List.iter (fun s -> print_endline (pretty (parse_string S_parser.start S_lexer.token s))) [ "vxy"; "vxz"; "wxy"; "wxz" ]

Observe that the middle `x` is recognized as an `a` in `vxy` and `wxz`, but as a `b` in `vxz` and `wxy`.